# Auditoría del dataset Chákṣu

Este notebook documenta la revisión inicial del dataset Chákṣu utilizado para
el entrenamiento del modelo especializado en riesgo de Glaucoma.

Objetivos de la auditoría:

1. Verificar la estructura de los archivos de decisión.
2. Comprobar dimensiones y columnas.
3. Identificar valores nulos o vacíos.
4. Identificar registros e imágenes duplicadas.
5. Revisar los valores originales de las decisiones de los expertos.
6. Verificar la consistencia de la decisión mayoritaria.
7. Analizar la distribución de las clases por dispositivo.
8. Contrastar las cantidades encontradas con la documentación oficial.
9. Detectar posibles inconsistencias antes de preparar el dataset experimental.

En una primera etapa se auditan los archivos de decisión del subconjunto Train. Posteriormente, se utilizan los subconjuntos físicos Train y Test para verificar la partición oficial y resolver posibles inconsistencias detectadas en los archivos de decisión. No se modifican las etiquetas ni los archivos fuente.

## 3. Importaciones

Importación de librerías esenciales (`pathlib` y `pandas`) y configuración de visualización de datos.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 4. Configurar las rutas

Definición de las rutas del proyecto, ubicación de las decisiones de Train y carpeta de resultados para las evidencias de la auditoría.

In [2]:
PROJECT_ROOT = Path(r"C:\Users\SeuNg720\Documents\TP1-RetiScan")

print("Raíz del proyecto:", PROJECT_ROOT)
print("¿Existe?:", PROJECT_ROOT.exists())

Raíz del proyecto: C:\Users\SeuNg720\Documents\TP1-RetiScan
¿Existe?: True


In [3]:
TRAIN_DECISIONS_DIR = PROJECT_ROOT / "datasets/glaucoma/chaksu/raw/Train/6.0_Glaucoma_Decision"
AUDIT_RESULTS_DIR = PROJECT_ROOT / "results/glaucoma/data_audit"

AUDIT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Carpeta de decisiones: {TRAIN_DECISIONS_DIR}")
print(f"¿Existe?: {TRAIN_DECISIONS_DIR.exists()}")

Carpeta de decisiones: C:\Users\SeuNg720\Documents\TP1-RetiScan\datasets\glaucoma\chaksu\raw\Train\6.0_Glaucoma_Decision
¿Existe?: True


## 5. Comprobar los archivos disponibles

Listado de los archivos CSV presentes en el subdirectorio de decisiones para registrar la evidencia de qué archivos originales utilizamos.

In [4]:
for archivo in sorted(TRAIN_DECISIONS_DIR.glob("*.csv")):
    print(archivo.name)

Glaucoma_Decision_Comparison_Bosch.csv
Glaucoma_Decision_Comparison_Bosch_majority.csv
Glaucoma_Decision_Comparison_Forus.csv
Glaucoma_Decision_Comparison_Forus_majority.csv
Glaucoma_Decision_Comparison_Remidio.csv
Glaucoma_Decision_Comparison_Remidio_majority.csv


## 6. Cargar los tres CSV que contienen Majority Decision

Para nuestro ground truth inicial nos interesan especialmente los archivos `_majority.csv` (Bosch, Forus y Remidio). Comprobamos sus dimensiones y nombres de columnas.

In [5]:
archivos_majority = {
    "Bosch": TRAIN_DECISIONS_DIR / "Glaucoma_Decision_Comparison_Bosch_majority.csv",
    "Forus": TRAIN_DECISIONS_DIR / "Glaucoma_Decision_Comparison_Forus_majority.csv",
    "Remidio": TRAIN_DECISIONS_DIR / "Glaucoma_Decision_Comparison_Remidio_majority.csv",
}
dfs = {
    camara: pd.read_csv(ruta)
    for camara, ruta in archivos_majority.items()
}

for camara, df in dfs.items():
    print(f"\n--- {camara} ---")
    print("Dimensiones:", df.shape)
    print("Columnas:")
    print(df.columns.tolist())


--- Bosch ---
Dimensiones: (145, 7)
Columnas:
['Images', 'Expert.1', 'Expert.2', 'Expert.3', 'Expert.4', 'Expert.5', 'Majority Decision']

--- Forus ---
Dimensiones: (95, 7)
Columnas:
['Images', 'Expert.1', 'Expert.2', 'Expert.3', 'Expert.4', 'Expert.5', 'Majority Decision']

--- Remidio ---
Dimensiones: (810, 7)
Columnas:
['Images', 'Expert.1', 'Expert.2', 'Expert.3', 'Expert.4', 'Expert.5', 'Majority Decision']


## 7. Visualizar los primeros registros

Inspección visual de las primeras filas de cada dataset para documentar el formato original de las etiquetas y detectar posibles variantes ortográficas en las decisiones de los expertos.

In [6]:
for camara, df in dfs.items():
    print(f"\n===== {camara} =====")
    display(df.head())


===== Bosch =====


,Images,Expert.1,Expert.2,Expert.3,Expert.4,Expert.5,Majority Decision
0,Image101.jpg-Image101-1.jpg,GLAUCOMA SUSPECT,GLAUCOMA SUSPECT,GLAUCOMA SUSPECT,GLAUCOMA SUSUPECT,GLAUCOMA SUSUPECT,GLAUCOMA SUSPECT
1,Image102.jpg-Image102-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
2,Image103.jpg-Image103-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
3,Image104.jpg-Image104-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
4,Image105.jpg-Image105-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL



===== Forus =====


,Images,Expert.1,Expert.2,Expert.3,Expert.4,Expert.5,Majority Decision
0,1.jpg-1-1.jpg,NORMAL,GLAUCOMA SUSPECT,GLAUCOMA SUSPECT,GLAUCOMA SUSUPECT,NORMAL,GLAUCOMA SUSPECT
1,10.jpg-10-1.jpg,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL
2,11.jpg-11-1.jpg,NORMAL,NORMAL,NORMAL,GLAUCOMA SUSUPECT,GLAUCOMA SUSUPECT,NORMAL
3,12.jpg-12-1.jpg,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL
4,13.jpg-13-1.jpg,GLAUCOMA SUSPECT,GLAUCOMA SUSPECT,NORMAL,GLAUCOMA SUSUPECT,NORMAL,GLAUCOMA SUSPECT



===== Remidio =====


,Images,Expert.1,Expert.2,Expert.3,Expert.4,Expert.5,Majority Decision
0,17521.tif-17521-1.tif,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
1,18499.tif-18499-1.tif,NORMAL,GLAUCOMA SUSPECT,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL
2,35977.tif-35977-1.tif,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL
3,37749.tif-37749-1.tif,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
4,39636.tif-39636-1.tif,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL,NORMAL


## 8. Comprobar valores nulos

Comprobación formal de valores nulos (`NaN`) en cada columna y guardado de la evidencia en `results/glaucoma/data_audit/nulos_train.csv`.

In [7]:
resumen_nulos = []
for camara, df in dfs.items():
    nulos = df.isna().sum()
    tabla = pd.DataFrame({
        "camara": camara,
        "columna": nulos.index,
        "valores_nulos": nulos.values
    })
    resumen_nulos.append(tabla)
resumen_nulos = pd.concat(resumen_nulos, ignore_index=True)
display(resumen_nulos)

resumen_nulos.to_csv(
    AUDIT_RESULTS_DIR / "nulos_train.csv",
    index=False
)

,camara,columna,valores_nulos
0,Bosch,Images,0
1,Bosch,Expert.1,0
2,Bosch,Expert.2,0
3,Bosch,Expert.3,0
4,Bosch,Expert.4,0
5,Bosch,Expert.5,0
6,Bosch,Majority Decision,0
7,Forus,Images,0
8,Forus,Expert.1,0
9,Forus,Expert.2,0


## 9. También comprobar valores vacíos

Dado que `NaN` y cadenas vacías no son lo mismo, validamos la presencia de campos con texto en blanco (`""`). Guardamos evidencia en `vacios_train.csv`.

In [8]:
resumen_vacios = []
for camara, df in dfs.items():
    for columna in df.columns:
        vacios = (
            df[columna]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
        resumen_vacios.append({
            "camara": camara,
            "columna": columna,
            "valores_vacios": vacios
        })
resumen_vacios = pd.DataFrame(resumen_vacios)
display(resumen_vacios)

resumen_vacios.to_csv(
    AUDIT_RESULTS_DIR / "vacios_train.csv",
    index=False
)

,camara,columna,valores_vacios
0,Bosch,Images,0
1,Bosch,Expert.1,0
2,Bosch,Expert.2,0
3,Bosch,Expert.3,0
4,Bosch,Expert.4,0
5,Bosch,Expert.5,0
6,Bosch,Majority Decision,0
7,Forus,Images,0
8,Forus,Expert.1,0
9,Forus,Expert.2,0


## 10. Comprobar filas duplicadas

Comprobación de filas duplicadas y nombres de archivo de imagen duplicados en la columna `Images`. Guardamos evidencia en `duplicados_train.csv`.

In [9]:
resumen_duplicados = []
for camara, df in dfs.items():
    filas_duplicadas = df.duplicated().sum()
    imagenes_duplicadas = df["Images"].duplicated().sum()
    resumen_duplicados.append({
        "camara": camara,
        "filas_duplicadas": filas_duplicadas,
        "imagenes_duplicadas": imagenes_duplicadas
    })
resumen_duplicados = pd.DataFrame(resumen_duplicados)
display(resumen_duplicados)

resumen_duplicados.to_csv(
    AUDIT_RESULTS_DIR / "duplicados_train.csv",
    index=False
)

,camara,filas_duplicadas,imagenes_duplicadas
0,Bosch,0,0
1,Forus,0,0
2,Remidio,0,0


## 11. Revisar los valores originales de las etiquetas

Auditoría de las categorías asignadas por cada experto y por la decisión mayoritaria. Se identifican inconsistencias ortográficas (p. ej. `GLAUCOMA SUSUPECT`) en anotaciones individuales, manteniendo la integridad del dato original.

In [10]:
columnas_etiquetas = [
    "Expert.1",
    "Expert.2",
    "Expert.3",
    "Expert.4",
    "Expert.5",
    "Majority Decision"
]
for camara, df in dfs.items():
    print(f"\n===== {camara} =====")
    for columna in columnas_etiquetas:
        print(f"\n{columna}:")
        print(df[columna].value_counts(dropna=False))


===== Bosch =====

Expert.1:
Expert.1
NORMAL              123
GLAUCOMA SUSPECT     22
Name: count, dtype: int64

Expert.2:
Expert.2
NORMAL              88
GLAUCOMA SUSPECT    57
Name: count, dtype: int64

Expert.3:
Expert.3
NORMAL              119
GLAUCOMA SUSPECT     26
Name: count, dtype: int64

Expert.4:
Expert.4
NORMAL                123
GLAUCOMA  SUSUPECT     22
Name: count, dtype: int64

Expert.5:
Expert.5
NORMAL                140
GLAUCOMA  SUSUPECT      5
Name: count, dtype: int64

Majority Decision:
Majority Decision
NORMAL              125
GLAUCOMA SUSPECT     20
Name: count, dtype: int64

===== Forus =====

Expert.1:
Expert.1
NORMAL              74
GLAUCOMA SUSPECT    21
Name: count, dtype: int64

Expert.2:
Expert.2
NORMAL              68
GLAUCOMA SUSPECT    27
Name: count, dtype: int64

Expert.3:
Expert.3
NORMAL              75
GLAUCOMA SUSPECT    20
Name: count, dtype: int64

Expert.4:
Expert.4
NORMAL                73
GLAUCOMA  SUSUPECT    22
Name: count, dtype: int64

E

## 12. Revisar específicamente Majority Decision

Distribución preliminar encontrada en los CSV suministrados dentro del paquete Train antes de resolver la inconsistencia observada en Bosch.

In [11]:
for camara, df in dfs.items():
    print(f"\n{camara}")
    print(
        df["Majority Decision"]
        .value_counts(dropna=False)
    )


Bosch
Majority Decision
NORMAL              125
GLAUCOMA SUSPECT     20
Name: count, dtype: int64

Forus
Majority Decision
NORMAL              79
GLAUCOMA SUSPECT    16
Name: count, dtype: int64

Remidio
Majority Decision
NORMAL              708
GLAUCOMA SUSPECT    102
Name: count, dtype: int64


## 13. Normalización únicamente para comprobar el voto

Función de normalización para comparar `GLAUCOMA SUSPECT` con el typo `GLAUCOMA SUSUPECT`. No altera los archivos fuente.

In [12]:
def normalizar_decision(valor):
    valor = str(valor).strip().upper()
    # Elimina espacios repetidos
    valor = " ".join(valor.split())
    correcciones = {
        "GLAUCOMA SUSUPECT": "GLAUCOMA SUSPECT"
    }
    return correcciones.get(valor, valor)

## 14. Recalcular nosotros mismos el voto mayoritario

Recálculo de la moda a partir de los 5 expertos para verificar que la columna `Majority Decision` coincide matemáticamente con el voto mayoritario. Guardamos evidencia en `consistencia_majority_train.csv`.

In [13]:
expert_cols = [
    "Expert.1",
    "Expert.2",
    "Expert.3",
    "Expert.4",
    "Expert.5"
]
resultados_consistencia = []
for camara, df_original in dfs.items():
    df = df_original.copy()
    # Normalizar únicamente para la comprobación
    for columna in expert_cols:
        df[columna + "_norm"] = df[columna].apply(normalizar_decision)
    df["majority_norm"] = (
        df["Majority Decision"]
        .apply(normalizar_decision)
    )
    expert_norm_cols = [
        columna + "_norm"
        for columna in expert_cols
    ]
    # Recalcular majority vote de los cinco expertos
    df["majority_calculada"] = (
        df[expert_norm_cols]
        .mode(axis=1)[0]
    )
    df["coincide_majority"] = (
        df["majority_calculada"]
        == df["majority_norm"]
    )
    total = len(df)
    coincidencias = df["coincide_majority"].sum()
    discrepancias = (~df["coincide_majority"]).sum()
    resultados_consistencia.append({
        "camara": camara,
        "total": total,
        "coincidencias": coincidencias,
        "discrepancias": discrepancias
    })

resumen_consistencia = pd.DataFrame(
    resultados_consistencia
)
display(resumen_consistencia)

resumen_consistencia.to_csv(
    AUDIT_RESULTS_DIR / "consistencia_majority_train.csv",
    index=False
)

,camara,total,coincidencias,discrepancias
0,Bosch,145,145,0
1,Forus,95,95,0
2,Remidio,810,810,0


## 15. Guardar posibles discrepancias

Extracción y registro formal de cualquier discrepancia detectada. Guardamos evidencia en `inconsistencias_majority_train.csv`.

In [14]:
inconsistencias = []
for camara, df_original in dfs.items():
    df = df_original.copy()
    for columna in expert_cols:
        df[columna + "_norm"] = df[columna].apply(normalizar_decision)
    df["majority_norm"] = (
        df["Majority Decision"]
        .apply(normalizar_decision)
    )
    expert_norm_cols = [
        columna + "_norm"
        for columna in expert_cols
    ]
    df["majority_calculada"] = (
        df[expert_norm_cols]
        .mode(axis=1)[0]
    )
    df["coincide"] = (
        df["majority_calculada"]
        == df["majority_norm"]
    )
    errores = df.loc[
        ~df["coincide"],
        [
            "Images",
            "Majority Decision",
            "majority_calculada"
        ]
    ].copy()
    errores.insert(0, "camara", camara)
    inconsistencias.append(errores)

inconsistencias = pd.concat(
    inconsistencias,
    ignore_index=True
)
display(inconsistencias)

inconsistencias.to_csv(
    AUDIT_RESULTS_DIR / "inconsistencias_majority_train.csv",
    index=False
)

,camara,Images,Majority Decision,majority_calculada


## 16. Ahora hacemos la comparación con el README

El README establece que Train contiene Bosch = 104, Forus = 95, Remidio = 810 (total 1,009 imágenes). Comprobamos automáticamente las cantidades y guardamos evidencia en `comparacion_cantidades_train.csv`.

In [15]:
esperado_train = {
    "Bosch": 104,
    "Forus": 95,
    "Remidio": 810
}
comparacion_cantidades = []
for camara, df in dfs.items():
    encontrado = len(df)
    esperado = esperado_train[camara]
    comparacion_cantidades.append({
        "camara": camara,
        "esperado_readme": esperado,
        "encontrado_csv": encontrado,
        "diferencia": encontrado - esperado
    })
comparacion_cantidades = pd.DataFrame(
    comparacion_cantidades
)
display(comparacion_cantidades)

comparacion_cantidades.to_csv(
    AUDIT_RESULTS_DIR / "comparacion_cantidades_train.csv",
    index=False
)

,camara,esperado_readme,encontrado_csv,diferencia
0,Bosch,104,145,41
1,Forus,95,95,0
2,Remidio,810,810,0


## Conclusiones preliminares de la auditoría de Train

- No se detectaron valores nulos en los archivos de decisión mayoritaria analizados.
- No se detectaron filas duplicadas ni identificadores de imagen duplicados dentro de cada archivo.
- Se identificaron variantes ortográficas en las decisiones individuales de algunos expertos, como
  `GLAUCOMA SUSUPECT`; estas no afectan directamente a la columna `Majority Decision`.
- La decisión mayoritaria proporcionada por Chákṣu fue recalculada a partir de los cinco expertos y
  no se identificaron discrepancias.
- Los archivos Forus y Remidio coinciden con las cantidades documentadas para el subconjunto Train.
- El archivo Bosch contiene 145 registros, mientras que la documentación indica 104 imágenes Bosch
  para Train y 41 para Test.
- La diferencia de 41 registros requiere comprobar los identificadores del subconjunto Test antes de
  definir las particiones experimentales.
- En esta etapa no se modificaron archivos ni etiquetas originales.

## 17. Rutas para comprobación de imágenes físicas y Test

Configuración de las rutas hacia los directorios de imágenes originales de Train y Test, así como las decisiones de Test para contrastar los archivos físicos contra los registros del CSV de Bosch.

In [16]:
TRAIN_IMAGES_DIR = PROJECT_ROOT / "datasets" / "glaucoma" / "chaksu" / "raw" / "Train" / "1.0_Original_Fundus_Images"
TEST_IMAGES_DIR = PROJECT_ROOT / "datasets" / "glaucoma" / "chaksu" / "raw" / "Test" / "1.0_Original_Fundus_Images"
TEST_DECISIONS_DIR = PROJECT_ROOT / "datasets" / "glaucoma" / "chaksu" / "raw" / "Test" / "6.0_Glaucoma_Decision"

print("Train images:", TRAIN_IMAGES_DIR.exists())
print("Test images:", TEST_IMAGES_DIR.exists())
print("Test decisions:", TEST_DECISIONS_DIR.exists())

Train images: True
Test images: True
Test decisions: True


## 18. Conteo físico de imágenes en Train y Test

Contamos físicamente los archivos de imagen en los directorios de Train y Test para verificar si corresponden a los totales oficiales documentados por Chákṣu (1,009 en Train y 336 en Test).

In [17]:
EXTENSIONES_IMAGEN = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff"
}

def obtener_imagenes(carpeta):
    return [
        archivo
        for archivo in carpeta.rglob("*")
        if archivo.is_file()
        and archivo.suffix.lower() in EXTENSIONES_IMAGEN
    ]

imagenes_train = obtener_imagenes(TRAIN_IMAGES_DIR)
imagenes_test = obtener_imagenes(TEST_IMAGES_DIR)

print("Imágenes físicas Train:", len(imagenes_train))
print("Imágenes físicas Test:", len(imagenes_test))

Imágenes físicas Train: 1009
Imágenes físicas Test: 336


## 19. Carga de la tabla Bosch

Cargamos el archivo `Glaucoma_Decision_Comparison_Bosch_majority.csv` para inspeccionar la estructura de la columna `Images`.

In [18]:
bosch_decisiones = pd.read_csv(
    TRAIN_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Bosch_majority.csv"
)
bosch_decisiones.head()

,Images,Expert.1,Expert.2,Expert.3,Expert.4,Expert.5,Majority Decision
0,Image101.jpg-Image101-1.jpg,GLAUCOMA SUSPECT,GLAUCOMA SUSPECT,GLAUCOMA SUSPECT,GLAUCOMA SUSUPECT,GLAUCOMA SUSUPECT,GLAUCOMA SUSPECT
1,Image102.jpg-Image102-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
2,Image103.jpg-Image103-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
3,Image104.jpg-Image104-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL
4,Image105.jpg-Image105-1.jpg,NORMAL,GLAUCOMA SUSPECT,NORMAL,NORMAL,NORMAL,NORMAL


## 20. Extracción del nombre original de la imagen

El campo `Images` tiene el patrón `NombreOriginal.jpg-NombreRecorte.jpg`. Extraemos el primer nombre para identificar el archivo fuente sin alterar los datos originales.

In [19]:
bosch_decisiones["imagen_original"] = (
    bosch_decisiones["Images"]
    .str.extract(
        r"^(.+?\.(?:jpg|jpeg|png|tif|tiff))-",
        expand=False
    )
)
bosch_decisiones[
    ["Images", "imagen_original"]
].head()

,Images,imagen_original
0,Image101.jpg-Image101-1.jpg,Image101.jpg
1,Image102.jpg-Image102-1.jpg,Image102.jpg
2,Image103.jpg-Image103-1.jpg,Image103.jpg
3,Image104.jpg-Image104-1.jpg,Image104.jpg
4,Image105.jpg-Image105-1.jpg,Image105.jpg


## 21. Clasificación de las 145 filas de Bosch según ubicación física

Obtenemos el conjunto de nombres de archivos físicos de Train y Test y clasificamos cada uno de los 145 registros de Bosch.

> **Nota:** Normalizamos a minúsculas (`.lower()`) para garantizar que la comparación sea insensible a mayúsculas/minúsculas en la extensión (por ejemplo `.JPG` en disco vs. `.jpg` en el CSV).

In [20]:
nombres_train = {
    archivo.name.lower()
    for archivo in imagenes_train
}
nombres_test = {
    archivo.name.lower()
    for archivo in imagenes_test
}

bosch_decisiones["esta_en_train"] = (
    bosch_decisiones["imagen_original"]
    .str.lower()
    .isin(nombres_train)
)
bosch_decisiones["esta_en_test"] = (
    bosch_decisiones["imagen_original"]
    .str.lower()
    .isin(nombres_test)
)

print(
    "Bosch localizadas en Train:",
    bosch_decisiones["esta_en_train"].sum()
)
print(
    "Bosch localizadas en Test:",
    bosch_decisiones["esta_en_test"].sum()
)
print(
    "Bosch presentes en ambos:",
    (
        bosch_decisiones["esta_en_train"]
        & bosch_decisiones["esta_en_test"]
    ).sum()
)
print(
    "Bosch no encontradas:",
    (
        ~bosch_decisiones["esta_en_train"]
        & ~bosch_decisiones["esta_en_test"]
    ).sum()
)

Bosch localizadas en Train: 104
Bosch localizadas en Test: 41
Bosch presentes en ambos: 0
Bosch no encontradas: 0


## 22. Evidencia formal de la resolución de la anomalía de Bosch

Construcción del DataFrame de resumen que evidencia la partición exacta de las 145 imágenes Bosch (104 en Train y 41 en Test) y guardado en `results/glaucoma/data_audit/verificacion_split_bosch.csv`.

In [21]:
resumen_bosch_split = pd.DataFrame({
    "concepto": [
        "Total decisiones Bosch",
        "Imágenes Bosch en Train",
        "Imágenes Bosch en Test",
        "Presentes en ambos",
        "No encontradas"
    ],
    "cantidad": [
        len(bosch_decisiones),
        bosch_decisiones["esta_en_train"].sum(),
        bosch_decisiones["esta_en_test"].sum(),
        (
            bosch_decisiones["esta_en_train"]
            & bosch_decisiones["esta_en_test"]
        ).sum(),
        (
            ~bosch_decisiones["esta_en_train"]
            & ~bosch_decisiones["esta_en_test"]
        ).sum()
    ]
})
display(resumen_bosch_split)

resumen_bosch_split.to_csv(
    AUDIT_RESULTS_DIR / "verificacion_split_bosch.csv",
    index=False
)

,concepto,cantidad
0,Total decisiones Bosch,145
1,Imágenes Bosch en Train,104
2,Imágenes Bosch en Test,41
3,Presentes en ambos,0
4,No encontradas,0


### Resolución de la inconsistencia Bosch

El archivo de decisiones de Bosch contiene 145 registros, correspondientes al
total de imágenes Bosch del dataset. Mediante la comparación entre los
identificadores de imagen del CSV y los archivos físicos de los subconjuntos
oficiales, se comprobó que:

- 104 imágenes pertenecen a Train.
- 41 imágenes pertenecen a Test.
- No existen imágenes Bosch presentes simultáneamente en ambos subconjuntos.
- No existen registros Bosch sin correspondencia con una imagen física.

Por tanto, la partición oficial puede reconstruirse de manera determinística
utilizando la ubicación física de cada retinografía.

## 23. Auditoría del subconjunto Test

En esta etapa se auditan las decisiones correspondientes al subconjunto Test
oficial de Chákṣu.

Para Bosch, el archivo de decisiones contiene las 145 imágenes adquiridas con
ese dispositivo, por lo que se seleccionan únicamente las 41 imágenes que se
encuentran físicamente en el subconjunto Test.

Para Forus y Remidio, los archivos de comparación contienen directamente las
31 y 264 imágenes correspondientes al Test oficial.

Los archivos originales ubicados en `raw/` no son modificados.

In [22]:
test_bosch_global = pd.read_csv(
    TEST_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Bosch_majority.csv"
)
test_forus = pd.read_csv(
    TEST_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Forus_majority.csv"
)
test_remidio = pd.read_csv(
    TEST_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Remidio_majority.csv"
)

print("Bosch CSV global:", test_bosch_global.shape)
print("Forus Test:", test_forus.shape)
print("Remidio Test:", test_remidio.shape)

Bosch CSV global: (145, 7)
Forus Test: (31, 7)
Remidio Test: (264, 7)


## 24. Obtener únicamente las 41 Bosch de Test

Selección determinística de las 41 imágenes correspondientes al subconjunto Test para Bosch a partir de la coincidencia con los archivos físicos en disco.

In [23]:
test_bosch = test_bosch_global.copy()
test_bosch["imagen_original"] = (
    test_bosch["Images"]
    .str.extract(
        r"^(.+?\.(?:jpg|jpeg|png|tif|tiff))-",
        expand=False
    )
)
test_bosch["imagen_original_norm"] = (
    test_bosch["imagen_original"]
    .str.lower()
)
nombres_test_norm = {
    archivo.name.lower()
    for archivo in imagenes_test
}
test_bosch = test_bosch[
    test_bosch["imagen_original_norm"].isin(nombres_test_norm)
].copy()

print("Bosch Test real:", len(test_bosch))

Bosch Test real: 41


## 25. Uniformizar el nombre de la etiqueta final

Normalización de esquema en memoria renombrando `Glaucoma Decision` a `Majority Decision` en Forus y Remidio para mantener un formato uniforme entre dispositivos sin alterar los archivos CSV originales.

In [24]:
test_forus = test_forus.rename(
    columns={"Glaucoma Decision": "Majority Decision"}
)
test_remidio = test_remidio.rename(
    columns={"Glaucoma Decision": "Majority Decision"}
)

for nombre, df in {
    "Bosch": test_bosch,
    "Forus": test_forus,
    "Remidio": test_remidio
}.items():
    print(nombre, df.columns.tolist())

Bosch ['Images', 'Expert.1', 'Expert.2', 'Expert.3', 'Expert.4', 'Expert.5', 'Majority Decision', 'imagen_original', 'imagen_original_norm']
Forus ['Images', 'Expert.1', 'Expert.2', 'Expert.3', 'Expert.4', 'Expert.5', 'Majority Decision']
Remidio ['Images', 'Expert.1', 'Expert.2', 'Expert.3', 'Expert.4', 'Expert.5', 'Majority Decision']


## 26. Crear colección de DataFrames de Test

Consolidación de los subconjuntos de Test y verificación de totales contra la documentación oficial (41 Bosch, 31 Forus, 264 Remidio, total 336). Evidencia guardada en `cantidades_test.csv`.

In [25]:
dfs_test = {
    "Bosch": test_bosch,
    "Forus": test_forus,
    "Remidio": test_remidio
}

resumen_cantidades_test = pd.DataFrame([
    {
        "camara": camara,
        "cantidad": len(df)
    }
    for camara, df in dfs_test.items()
])
display(resumen_cantidades_test)
print(
    "Total Test:",
    resumen_cantidades_test["cantidad"].sum()
)

resumen_cantidades_test.to_csv(
    AUDIT_RESULTS_DIR / "cantidades_test.csv",
    index=False
)

,camara,cantidad
0,Bosch,41
1,Forus,31
2,Remidio,264


Total Test: 336


## 27. Nulos en Test

Verificación formal de la presencia de valores nulos o `NaN` en los subconjuntos de prueba de cada dispositivo y exportación de evidencia en `nulos_test.csv`.

In [26]:
resumen_nulos_test = []
for camara, df in dfs_test.items():
    nulos = df.isna().sum()
    tabla = pd.DataFrame({
        "camara": camara,
        "columna": nulos.index,
        "valores_nulos": nulos.values
    })
    resumen_nulos_test.append(tabla)
resumen_nulos_test = pd.concat(
    resumen_nulos_test,
    ignore_index=True
)
display(resumen_nulos_test)

resumen_nulos_test.to_csv(
    AUDIT_RESULTS_DIR / "nulos_test.csv",
    index=False
)

,camara,columna,valores_nulos
0,Bosch,Images,0
1,Bosch,Expert.1,0
2,Bosch,Expert.2,0
3,Bosch,Expert.3,0
4,Bosch,Expert.4,0
5,Bosch,Expert.5,0
6,Bosch,Majority Decision,0
7,Bosch,imagen_original,0
8,Bosch,imagen_original_norm,0
9,Forus,Images,0


## 28. Valores vacíos en Test

Detección de cadenas de texto vacías (`""`) en las columnas de los subconjuntos de Test y exportación de evidencia en `vacios_test.csv`.

In [27]:
resumen_vacios_test = []
for camara, df in dfs_test.items():
    for columna in df.columns:
        vacios = (
            df[columna]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
        resumen_vacios_test.append({
            "camara": camara,
            "columna": columna,
            "valores_vacios": vacios
        })
resumen_vacios_test = pd.DataFrame(
    resumen_vacios_test
)
display(resumen_vacios_test)

resumen_vacios_test.to_csv(
    AUDIT_RESULTS_DIR / "vacios_test.csv",
    index=False
)

,camara,columna,valores_vacios
0,Bosch,Images,0
1,Bosch,Expert.1,0
2,Bosch,Expert.2,0
3,Bosch,Expert.3,0
4,Bosch,Expert.4,0
5,Bosch,Expert.5,0
6,Bosch,Majority Decision,0
7,Bosch,imagen_original,0
8,Bosch,imagen_original_norm,0
9,Forus,Images,0


## 29. Duplicados dentro de Test

Evaluación de duplicados a nivel de filas y nombres de imagen en el subconjunto Test. Evidencia guardada en `duplicados_test.csv`.

In [28]:
resumen_duplicados_test = []
for camara, df in dfs_test.items():
    resumen_duplicados_test.append({
        "camara": camara,
        "filas_duplicadas": df.duplicated().sum(),
        "imagenes_duplicadas": df["Images"].duplicated().sum()
    })
resumen_duplicados_test = pd.DataFrame(
    resumen_duplicados_test
)
display(resumen_duplicados_test)

resumen_duplicados_test.to_csv(
    AUDIT_RESULTS_DIR / "duplicados_test.csv",
    index=False
)

,camara,filas_duplicadas,imagenes_duplicadas
0,Bosch,0,0
1,Forus,0,0
2,Remidio,0,0


## 30. Distribución real de las clases de Test

Análisis de la distribución de frecuencias para la decisión mayoritaria por dispositivo en el subconjunto Test. Evidencia exportada a `distribucion_clases_test.csv`.

In [29]:
distribucion_test = []
for camara, df in dfs_test.items():
    conteo = (
        df["Majority Decision"]
        .value_counts()
    )
    for etiqueta, cantidad in conteo.items():
        distribucion_test.append({
            "camara": camara,
            "etiqueta_original": etiqueta,
            "cantidad": cantidad
        })
distribucion_test = pd.DataFrame(
    distribucion_test
)
display(distribucion_test)

distribucion_test.to_csv(
    AUDIT_RESULTS_DIR / "distribucion_clases_test.csv",
    index=False
)

,camara,etiqueta_original,cantidad
0,Bosch,NORMAL,40
1,Bosch,GLAUCOMA SUSPECT,1
2,Forus,NORMAL,29
3,Forus,GLAUCOMA SUSPECT,2
4,Remidio,NORMAL,216
5,Remidio,GLAUCOMA SUSPECT,48


## 31. Comprobar nuevamente el majority voting en Test

Recálculo de la moda a partir de las decisiones individuales de los cinco expertos (Expert.1 a Expert.5) normalizadas para verificar la consistencia matemática de `Majority Decision`. Evidencia en `consistencia_majority_test.csv`.

In [30]:
expert_cols = [
    "Expert.1",
    "Expert.2",
    "Expert.3",
    "Expert.4",
    "Expert.5"
]

resultados_consistencia_test = []
for camara, df_original in dfs_test.items():
    df = df_original.copy()
    for columna in expert_cols:
        df[columna + "_norm"] = (
            df[columna]
            .apply(normalizar_decision)
        )
    df["majority_norm"] = (
        df["Majority Decision"]
        .apply(normalizar_decision)
    )
    expert_norm_cols = [
        columna + "_norm"
        for columna in expert_cols
    ]
    df["majority_calculada"] = (
        df[expert_norm_cols]
        .mode(axis=1)[0]
    )
    df["coincide_majority"] = (
        df["majority_calculada"]
        == df["majority_norm"]
    )
    resultados_consistencia_test.append({
        "camara": camara,
        "total": len(df),
        "coincidencias": df["coincide_majority"].sum(),
        "discrepancias": (~df["coincide_majority"]).sum()
    })

resumen_consistencia_test = pd.DataFrame(
    resultados_consistencia_test
)
display(resumen_consistencia_test)

resumen_consistencia_test.to_csv(
    AUDIT_RESULTS_DIR / "consistencia_majority_test.csv",
    index=False
)

,camara,total,coincidencias,discrepancias
0,Bosch,41,41,0
1,Forus,31,31,0
2,Remidio,264,264,0


## 32. Ahora una comprobación MUY importante: Train vs Test

Comprobación para demostrar que ninguna imagen física de todo Chákṣu aparece simultáneamente en Train y Test, eliminando el riesgo de fuga de datos (*data leakage*). Evidencia guardada en `duplicados_train_test.csv`.

In [31]:
nombres_train_norm = {
    archivo.name.lower()
    for archivo in imagenes_train
}
nombres_test_norm = {
    archivo.name.lower()
    for archivo in imagenes_test
}

interseccion_train_test = (
    nombres_train_norm
    & nombres_test_norm
)
print(
    "Imágenes repetidas entre Train y Test:",
    len(interseccion_train_test)
)

pd.DataFrame(
    sorted(interseccion_train_test),
    columns=["imagen"]
).to_csv(
    AUDIT_RESULTS_DIR / "duplicados_train_test.csv",
    index=False
)

Imágenes repetidas entre Train y Test: 0


## 33. Verificar las 1,345 imágenes físicas

Comprobación del total consolidado de imágenes físicas de Train y Test contra la cifra oficial de Chákṣu (1,345 imágenes).

In [32]:
total_imagenes_fisicas = (
    len(imagenes_train)
    + len(imagenes_test)
)
print("Train:", len(imagenes_train))
print("Test:", len(imagenes_test))
print("Total:", total_imagenes_fisicas)

Train: 1009
Test: 336
Total: 1345


## 34. Reconstrucción del subconjunto Train real de Bosch

El archivo de decisiones Bosch contiene las 145 imágenes adquiridas con este
dispositivo. A partir de la ubicación física de las imágenes se seleccionan
únicamente las 104 correspondientes al subconjunto Train oficial.

In [33]:
bosch_train = bosch_decisiones.copy()
bosch_train["imagen_original_norm"] = (
    bosch_train["imagen_original"]
    .str.lower()
)
bosch_train = bosch_train[
    bosch_train["imagen_original_norm"].isin(nombres_train_norm)
].copy()
print("Bosch Train real:", len(bosch_train))

Bosch Train real: 104


## 35. Construir el Train oficial correcto

Consolidación de los subconjuntos oficiales de Train para los tres dispositivos (Bosch: 104, Forus: 95, Remidio: 810, Total: 1,009) y guardado de evidencia en `cantidades_train_final.csv`.

In [34]:
dfs_train_real = {
    "Bosch": bosch_train,
    "Forus": dfs["Forus"].copy(),
    "Remidio": dfs["Remidio"].copy()
}

resumen_train_real = pd.DataFrame([
    {
        "camara": camara,
        "cantidad": len(df)
    }
    for camara, df in dfs_train_real.items()
])
display(resumen_train_real)
print(
    "Total Train oficial:",
    resumen_train_real["cantidad"].sum()
)

resumen_train_real.to_csv(
    AUDIT_RESULTS_DIR / "cantidades_train_final.csv",
    index=False
)

,camara,cantidad
0,Bosch,104
1,Forus,95
2,Remidio,810


Total Train oficial: 1009


## 36. Distribución real de clases de Train

Cálculo de la distribución de frecuencias para la decisión mayoritaria en el subconjunto Train real y exportación de evidencia a `distribucion_clases_train_final.csv`.

In [35]:
distribucion_train_real = []
for camara, df in dfs_train_real.items():
    conteo = (
        df["Majority Decision"]
        .value_counts()
    )
    for etiqueta, cantidad in conteo.items():
        distribucion_train_real.append({
            "camara": camara,
            "etiqueta_original": etiqueta,
            "cantidad": cantidad
        })
distribucion_train_real = pd.DataFrame(
    distribucion_train_real
)
display(distribucion_train_real)

distribucion_train_real.to_csv(
    AUDIT_RESULTS_DIR / "distribucion_clases_train_final.csv",
    index=False
)

,camara,etiqueta_original,cantidad
0,Bosch,NORMAL,85
1,Bosch,GLAUCOMA SUSPECT,19
2,Forus,NORMAL,79
3,Forus,GLAUCOMA SUSPECT,16
4,Remidio,NORMAL,708
5,Remidio,GLAUCOMA SUSPECT,102


## 37. Distribución consolidada Train/Test

Consolidación global de las frecuencias de clases por dispositivo y partición (Train vs. Test), junto con el resumen agrupado por split.

In [36]:
def obtener_totales_clase(dfs_split, split):
    registros = []
    for camara, df in dfs_split.items():
        conteo = (
            df["Majority Decision"]
            .value_counts()
        )
        for etiqueta, cantidad in conteo.items():
            registros.append({
                "split": split,
                "camara": camara,
                "etiqueta_original": etiqueta,
                "cantidad": cantidad
            })
    return pd.DataFrame(registros)

distribucion_completa = pd.concat(
    [
        obtener_totales_clase(
            dfs_train_real,
            "train"
        ),
        obtener_totales_clase(
            dfs_test,
            "test"
        )
    ],
    ignore_index=True
)
display(distribucion_completa)

resumen_por_split_clase = (
    distribucion_completa
    .groupby(
        ["split", "etiqueta_original"],
        as_index=False
    )["cantidad"]
    .sum()
)
display(resumen_por_split_clase)

,split,camara,etiqueta_original,cantidad
0,train,Bosch,NORMAL,85
1,train,Bosch,GLAUCOMA SUSPECT,19
2,train,Forus,NORMAL,79
3,train,Forus,GLAUCOMA SUSPECT,16
4,train,Remidio,NORMAL,708
5,train,Remidio,GLAUCOMA SUSPECT,102
6,test,Bosch,NORMAL,40
7,test,Bosch,GLAUCOMA SUSPECT,1
8,test,Forus,NORMAL,29
9,test,Forus,GLAUCOMA SUSPECT,2


,split,etiqueta_original,cantidad
0,test,GLAUCOMA SUSPECT,51
1,test,NORMAL,285
2,train,GLAUCOMA SUSPECT,137
3,train,NORMAL,872


## 38. Inspección de patrones de nombres de imagen en Train

Visualización de los primeros registros de la columna `Images` para Bosch, Forus y Remidio en Train para comprobar el patrón de concatenación de nombres antes de construir la función de extracción general.

In [37]:
for camara, df in dfs_train_real.items():
    print(f"\n===== {camara} =====")
    display(
        df[["Images"]].head(10)
    )


===== Bosch =====


,Images
0,Image101.jpg-Image101-1.jpg
1,Image102.jpg-Image102-1.jpg
2,Image103.jpg-Image103-1.jpg
3,Image104.jpg-Image104-1.jpg
4,Image105.jpg-Image105-1.jpg
5,Image106.jpg-Image106-1.jpg
6,Image107.jpg-Image107-1.jpg
7,Image108.jpg-Image108-1.jpg
8,Image109.jpg-Image109-1.jpg
9,Image110.jpg-Image110-1.jpg



===== Forus =====


,Images
0,1.jpg-1-1.jpg
1,10.jpg-10-1.jpg
2,11.jpg-11-1.jpg
3,12.jpg-12-1.jpg
4,13.jpg-13-1.jpg
5,14.jpg-14-1.jpg
6,15.jpg-15-1.jpg
7,16.jpg-16-1.jpg
8,17.jpg-17-1.jpg
9,18.jpg-18-1.jpg



===== Remidio =====


,Images
0,17521.tif-17521-1.tif
1,18499.tif-18499-1.tif
2,35977.tif-35977-1.tif
3,37749.tif-37749-1.tif
4,39636.tif-39636-1.tif
5,46109.tif-46109-1.tif
6,51763.tif-51763-1.tif
7,64744.tif-64744-1.tif
8,66557.tif-66557-1.tif
9,67239.tif-67239-1.tif


## 39. Inspección de patrones de nombres de imagen en Test

Visualización de los primeros registros de la columna `Images` para Bosch, Forus y Remidio en Test para verificar que sigan el mismo formato de concatenación.

In [38]:
for camara, df in dfs_test.items():
    print(f"\n===== {camara} TEST =====")
    display(
        df[["Images"]].head(10)
    )


===== Bosch TEST =====


,Images
86,P10_Image2.jpg-P10_Image2-1.jpg
87,P11_Image1.jpg-P11_Image1-1.jpg
88,P11_Image2.jpg-P11_Image2-1.jpg
89,P12_Image1.jpg-P12_Image1-1.jpg
90,P12_Image2.jpg-P12_Image2-1.jpg
91,P13_Image1.jpg-P13_Image1-1.jpg
92,P13_Image2.jpg-P13_Image2-1.jpg
93,P14_Image1.jpg-P14_Image1-1.jpg
94,P14_Image2.jpg-P14_Image2-1.jpg
95,P15_Image1.jpg-P15_Image1-1.jpg



===== Forus TEST =====


,Images
0,100.jpg-100-1.jpg
1,101.jpg-101-1.jpg
2,102.jpg-102-1.jpg
3,103.jpg-103-1.jpg
4,104.jpg-104-1.jpg
5,105.jpg-105-1.jpg
6,106.jpg-106-1.jpg
7,107.jpg-107-1.jpg
8,108.jpg-108-1.jpg
9,109.jpg-109-1.jpg



===== Remidio TEST =====


,Images
0,IMG_3709.tif-IMG_3709-1.tif
1,IMG_3710.tif-IMG_3710-1.tif
2,IMG_3711.tif-IMG_3711-1.tif
3,IMG_3713.tif-IMG_3713-1.tif
4,IMG_3714.tif-IMG_3714-1.tif
5,IMG_3719.tif-IMG_3719-1.tif
6,IMG_3720.tif-IMG_3720-1.tif
7,IMG_3727.tif-IMG_3727-1.tif
8,IMG_3728.tif-IMG_3728-1.tif
9,IMG_3729.tif-IMG_3729-1.tif


## 40. Preparar Train y Test con nombre original

Homogeneización de los subconjuntos Train y Test añadiendo los campos `camera`, `image_name`, `image_name_norm` y `official_split`.

> **Nota metodológica:** Los CSV originales de Chákṣu registran extensiones nominales que en algunos dispositivos no coinciden con el archivo físico almacenado en disco (p. ej., Forus referencia `.jpg` en el CSV pero los archivos físicos son `.png`; Remidio referencia `.tif` en el CSV pero los archivos físicos son `.jpg`/`.JPG`). Para asegurar una correspondencia física 100% válida, resolvemos el nombre exacto del archivo físico a partir del identificador base (*stem*).

In [39]:
stems_fisicos = [
    f.stem.lower()
    for f in imagenes_train + imagenes_test
]
print("Archivos físicos:", len(stems_fisicos))
print("Stems únicos:", len(set(stems_fisicos)))
print(
    "Stems duplicados:",
    len(stems_fisicos) - len(set(stems_fisicos))
)

mapa_archivos_fisicos = {
    f.stem.lower(): f.name
    for f in imagenes_train + imagenes_test
}

tablas_train = []
for camara, df_original in dfs_train_real.items():
    df = df_original.copy()
    df["camera"] = camara
    df["image_csv"] = (
        df["Images"]
        .str.extract(
            r"^(.+?\.(?:jpg|jpeg|png|tif|tiff))-",
            expand=False
        )
    )
    df["image_stem"] = (
        df["image_csv"]
        .str.replace(r"\.[^.]+$", "", regex=True)
        .str.lower()
    )
    df["image_name"] = df["image_stem"].map(mapa_archivos_fisicos)
    df["image_name_norm"] = df["image_name"].str.lower()
    df["official_split"] = "train"
    tablas_train.append(df)

tablas_test = []
for camara, df_original in dfs_test.items():
    df = df_original.copy()
    df["camera"] = camara
    df["image_csv"] = (
        df["Images"]
        .str.extract(
            r"^(.+?\.(?:jpg|jpeg|png|tif|tiff))-",
            expand=False
        )
    )
    df["image_stem"] = (
        df["image_csv"]
        .str.replace(r"\.[^.]+$", "", regex=True)
        .str.lower()
    )
    df["image_name"] = df["image_stem"].map(mapa_archivos_fisicos)
    df["image_name_norm"] = df["image_name"].str.lower()
    df["official_split"] = "test"
    tablas_test.append(df)

labels_train = pd.concat(
    tablas_train,
    ignore_index=True
)
labels_test = pd.concat(
    tablas_test,
    ignore_index=True
)

print("Etiquetas Train:", len(labels_train))
print("Etiquetas Test:", len(labels_test))
print("Total etiquetas:", len(labels_train) + len(labels_test))

Archivos físicos: 1345
Stems únicos: 1345
Stems duplicados: 0
Etiquetas Train: 1009
Etiquetas Test: 336
Total etiquetas: 1345


## 41. Comparar imágenes físicas vs etiquetas

Verificación formal de correspondencia biunívoca entre las imágenes físicas en disco y los registros de etiquetas en Train y Test. Guardamos evidencia en `correspondencia_imagenes_etiquetas.csv`.

In [40]:
imagenes_train_norm = {
    archivo.name.lower()
    for archivo in imagenes_train
}
imagenes_test_norm = {
    archivo.name.lower()
    for archivo in imagenes_test
}
etiquetas_train_norm = set(
    labels_train["image_name_norm"]
)
etiquetas_test_norm = set(
    labels_test["image_name_norm"]
)

imagenes_train_sin_etiqueta = (
    imagenes_train_norm
    - etiquetas_train_norm
)
etiquetas_train_sin_imagen = (
    etiquetas_train_norm
    - imagenes_train_norm
)
imagenes_test_sin_etiqueta = (
    imagenes_test_norm
    - etiquetas_test_norm
)
etiquetas_test_sin_imagen = (
    etiquetas_test_norm
    - imagenes_test_norm
)

print(
    "Train - imágenes sin etiqueta:",
    len(imagenes_train_sin_etiqueta)
)
print(
    "Train - etiquetas sin imagen:",
    len(etiquetas_train_sin_imagen)
)
print(
    "Test - imágenes sin etiqueta:",
    len(imagenes_test_sin_etiqueta)
)
print(
    "Test - etiquetas sin imagen:",
    len(etiquetas_test_sin_imagen)
)

resumen_correspondencia = pd.DataFrame([
    {
        "conjunto": "Train",
        "imagenes_sin_etiqueta": len(imagenes_train_sin_etiqueta),
        "etiquetas_sin_imagen": len(etiquetas_train_sin_imagen)
    },
    {
        "conjunto": "Test",
        "imagenes_sin_etiqueta": len(imagenes_test_sin_etiqueta),
        "etiquetas_sin_imagen": len(etiquetas_test_sin_imagen)
    }
])
display(resumen_correspondencia)

resumen_correspondencia.to_csv(
    AUDIT_RESULTS_DIR / "correspondencia_imagenes_etiquetas.csv",
    index=False
)

Train - imágenes sin etiqueta: 0
Train - etiquetas sin imagen: 0
Test - imágenes sin etiqueta: 0
Test - etiquetas sin imagen: 0


,conjunto,imagenes_sin_etiqueta,etiquetas_sin_imagen
0,Train,0,0
1,Test,0,0


## 42. Comprobar unicidad global y clave de identificación (`image_key`)

Evaluación de la unicidad global y definición de `image_key` (`<camera>::<image_name_norm>`), garantizando que cada registro del dataset esté inequívocamente indexado para el dataset maestro.

In [41]:
labels_all = pd.concat(
    [labels_train, labels_test],
    ignore_index=True
)
print(
    "Etiquetas totales:",
    len(labels_all)
)
print(
    "Nombres únicos:",
    labels_all["image_name_norm"].nunique()
)
print(
    "Duplicados globales:",
    labels_all["image_name_norm"]
    .duplicated()
    .sum()
)

labels_all["image_key"] = (
    labels_all["camera"].str.lower()
    + "::"
    + labels_all["image_name_norm"]
)

print(
    "Image keys únicas:",
    labels_all["image_key"].nunique()
)
print(
    "Duplicados image_key:",
    labels_all["image_key"]
    .duplicated()
    .sum()
)

Etiquetas totales: 1345
Nombres únicos: 1345
Duplicados globales: 0
Image keys únicas: 1345
Duplicados image_key: 0


# Conclusiones de la auditoría de Chákṣu

La auditoría permitió verificar la integridad estructural de los subconjuntos
Train y Test utilizados para la preparación experimental del modelo de riesgo
de Glaucoma.

Se comprobó que:

- El subconjunto Train contiene 1,009 retinografías y el subconjunto Test
  contiene 336, para un total de 1,345 imágenes.
- No existen imágenes repetidas entre los subconjuntos oficiales Train y Test.
- Cada retinografía posee exactamente una etiqueta correspondiente y no se
  identificaron etiquetas sin una imagen física asociada.
- No se detectaron registros duplicados ni valores nulos relevantes en los
  archivos de decisión analizados.
- La decisión mayoritaria proporcionada por Chákṣu es consistente con el voto
  recalculado de los cinco expertos en todos los registros analizados.
- Se identificaron variantes ortográficas en algunas decisiones individuales
  de expertos, sin afectar la etiqueta consolidada utilizada como ground truth.
- El archivo de decisiones Bosch contiene las 145 imágenes de dicho
  dispositivo. Mediante la correspondencia con los archivos físicos se
  reconstruyó correctamente la partición oficial de 104 imágenes Train y
  41 imágenes Test.
- La distribución final corresponde a 872 imágenes NORMAL y 137 GLAUCOMA
  SUSPECT en Train, y 285 NORMAL y 51 GLAUCOMA SUSPECT en Test.
- El conjunto presenta un desbalance hacia la clase NORMAL, aspecto que deberá
  ser considerado posteriormente durante el diseño del protocolo experimental.
- Los archivos originales contenidos en `raw/` no fueron modificados durante
  el proceso de auditoría.

Con estas verificaciones, el dataset queda estructuralmente validado para la
posterior construcción del conjunto maestro y la preparación de los datos de
entrenamiento, validación y prueba.